## 0. Task

Each official sentence gets one of 13 rhetorical roles. Metric is **macro-F1**, so rare roles (Dissent, PrecedentOverruled) dominate the number.

Selected: **efficient LoRMA** sentence encoder + **BiLSTM + CRF** over the sentence sequence.


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
SEARCH = [HERE, HERE.parent, Path("/kaggle/working")]
for base in (HERE, HERE.parent, HERE.parent.parent, HERE.parent.parent.parent):
    SEARCH.append(base)
    for phase in range(1, 8):
        SEARCH.append(base / f"phase{phase}")
        SEARCH.append(base / f"phase{phase}" / "code")
        SEARCH.append(base / "code" / f"phase{phase}")
for folder in SEARCH:
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
print("python path heads:")
for folder in sys.path[:8]:
    print(" ", folder)


## 1. Why a sentence-level CRF, not token tags

The unit is the official sentence list. The Phase-1 model mean-pools each sentence. A BiLSTM then reads the whole judgment so Fact vs Argument can use left/right sentences.


In [ ]:
import inspect
from phase3_rr_own import RoleCRF, RoleTagger

print(inspect.getsource(RoleTagger.__init__))


## 2. Role CRF

Unlike NER, any role may follow any role. Start/end vectors plus a full 13×13 transition matrix.


In [ ]:
print(inspect.getsource(RoleCRF))


## 3. Rarity bias

Dissent and PrecedentOverruled are almost absent. A positive bias on those CRF emissions is what moved the selected zip.


In [ ]:
print("selected : efficient + LoRMA + BiLSTM-CRF + rarity bias")
print("also run : hybrid LoRA, decoder IA3, encdec distillation")
print("script   : phase3_rr_own.py")


## 4. Zip


In [ ]:
print("prediction.json: {doc_id: [role, role, ...]}  one role per official sentence")
